# 1. La place de la France dans l'IA : retard et opacité

Données : base *Notable AI Models* d'Epoch AI (une ligne par modèle d'IA jugé notable).

Ce notebook répond à deux questions :

- **Partie A — le retard.** Combien de modèles notables et de modèles de pointe la France produit-elle, et à quel niveau de puissance de calcul, comparée aux États-Unis, à la Chine, au Royaume-Uni, à l'Allemagne et au Canada ?
- **Partie B — l'opacité.** Les développeurs publient-ils de moins en moins d'informations sur leurs modèles ?

**Hypothèses posées avant l'analyse**

1. La France produit peu de modèles de pointe, et l'écart de puissance de calcul avec les États-Unis se mesure en années.
2. Sa présence repose sur un très petit nombre d'acteurs.
3. La part de modèles dont la puissance de calcul est publiée diminue depuis 2020, surtout pour les modèles fermés.
4. Les modèles aux poids ouverts sont mieux documentés que les modèles fermés.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from config import FIGURES
from style import COULEURS_PAYS, ORDRE_PAYS, appliquer_style
appliquer_style()
FIGURES.mkdir(exist_ok=True)

## 0. Chargement et premier contrôle

In [ ]:
from nettoyage import charger_dernier, preparer_epoch, eclater_pays, annees_de_retard, INFOS_PUBLIABLES

brut = charger_dernier("epoch_notable_models")
print(brut.shape)
brut.head(3)

In [ ]:
epoch = preparer_epoch(brut)
print(f"{len(brut) - len(epoch)} modèles écartés faute de date de publication")
print(f"Période couverte : {epoch['annee'].min()} - {epoch['annee'].max()}")
epoch[["Model", "Organization", "Country (of organization)", "annee", "ouvert", "frontiere", "score_transparence"]].head()

### Taux de remplissage des variables

Avant toute analyse : quelles colonnes sont exploitables ? C'est aussi le premier résultat de la partie B — une valeur manquante peut signifier que le développeur n'a pas publié l'information.

In [ ]:
colonnes = ["Organization", "Country (of organization)", "Publication date", "Parameters",
            "Training compute (FLOP)", "Training dataset size (total)",
            "Training hardware", "Training compute cost (2023 USD)"]
remplissage = brut[colonnes].notna().mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(remplissage.index, remplissage.values, color="#2a78d6", height=0.6)
for y, v in enumerate(remplissage.values):
    ax.text(v + 0.01, y, f"{v:.0%}", va="center", color="#52514e", fontsize=9)
ax.set_xlim(0, 1.1)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_title("Part des modèles pour lesquels chaque information est renseignée")
ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(FIGURES / "01_remplissage.png")

> **À interpréter.** Quelles variables sont assez complètes pour être analysées telles quelles ? Lesquelles ne pourront servir que comme indicateur d'opacité ? Pourquoi le coût d'entraînement ne peut-il pas être une variable centrale ?

## Partie A — Le retard de la France

### A.1 Pays des modèles

Un modèle co-développé par plusieurs pays est réparti entre eux à parts égales (**comptage fractionnaire**). Choix alternatif à discuter : compter le modèle entièrement pour chaque pays.

In [ ]:
par_pays = eclater_pays(epoch)

print("Collaborations internationales :", (epoch["Country (of organization)"].str.contains(",", na=False)).sum())
synthese = (par_pays.groupby("pays")
            .agg(modeles=("poids", "sum"),
                 dont_frontiere=("poids", lambda p: p[par_pays.loc[p.index, "frontiere"]].sum()),
                 part_ouverts=("ouvert", lambda o: o.astype("float").mean()),
                 premiere_annee=("annee", "min"))
            .reindex(ORDRE_PAYS).round(2))
synthese["part_du_total"] = (synthese["modeles"] / synthese["modeles"].sum()).round(3)
synthese

> **À interpréter.** Quel est le rang de la France ? Quelle part des modèles français sont des modèles de pointe, comparé aux États-Unis et à la Chine ? La part de modèles ouverts confirme-t-elle la stratégie « ouverte » associée à la France ?

### A.2 Évolution du nombre de modèles notables

In [ ]:
compte = (par_pays.groupby(["annee", "pays"])["poids"].sum()
          .unstack(fill_value=0).reindex(columns=ORDRE_PAYS, fill_value=0))
compte = compte.loc[compte.index >= 2015]

fig, ax = plt.subplots()
for pays in ORDRE_PAYS:
    if pays == "Autres":
        continue
    ax.plot(compte.index, compte[pays], color=COULEURS_PAYS[pays], label=pays,
            linewidth=3 if pays == "France" else 1.8)
ax.set_title("Modèles d'IA notables publiés chaque année, par pays")
ax.set_ylabel("Nombre de modèles (comptage fractionnaire)")
ax.legend(ncol=3, loc="upper left")
fig.tight_layout(); fig.savefig(FIGURES / "02_modeles_par_an.png")

In [ ]:
# Part de chaque pays, par période de 3 ans : plus lisible que des comptes annuels
par_pays["periode"] = pd.cut(par_pays["annee"], bins=[2014, 2017, 2020, 2023, 2026],
                             labels=["2015-17", "2018-20", "2021-23", "2024-26"])
parts = (par_pays.groupby(["periode", "pays"], observed=True)["poids"].sum()
         .groupby(level=0).transform(lambda s: s / s.sum()).unstack()
         .reindex(columns=ORDRE_PAYS).fillna(0))
parts.style.format("{:.1%}")

> **À interpréter.** La part de la France progresse-t-elle ou recule-t-elle ? À quelle période apparaît-elle vraiment ? Que devient la part de la Chine ?

### A.3 Puissance de calcul : la course à l'échelle

La puissance de calcul (en FLOP) varie sur plusieurs ordres de grandeur : on la représente en **échelle logarithmique**. Les seuls modèles présents ici sont ceux dont la puissance de calcul est connue (voir partie B pour le biais que cela introduit).

In [ ]:
calc = par_pays.dropna(subset=["log10_calcul"])
calc = calc.loc[calc["annee"] >= 2015]

fig, ax = plt.subplots()
for pays in ["Autres", "Royaume-Uni", "Allemagne", "Canada", "Chine", "États-Unis", "France"]:
    sous = calc.loc[calc["pays"] == pays]
    gris = pays not in {"France", "États-Unis", "Chine"}
    ax.scatter(sous["date"], sous["log10_calcul"], s=18 if gris else 30,
               color="#d6d5d0" if gris else COULEURS_PAYS[pays],
               edgecolor="white", linewidth=0.6, label=None if gris else pays, zorder=1 if gris else 3)
ax.set_title("Puissance de calcul d'entraînement des modèles notables")
ax.set_ylabel("log10(FLOP)")
ax.legend(title="Gris : autres pays", loc="upper left")
fig.tight_layout(); fig.savefig(FIGURES / "03_calcul_nuage.png")

### A.4 Temps de doublement de la puissance de calcul

Régression log-linéaire : $\log_{10}(\text{calcul}) = \alpha + \beta \times \text{date} + \varepsilon$.
Le temps de doublement vaut $\log_{10}(2) / \beta$ (en années). On l'estime pour tous les modèles, puis pour les seuls modèles de pointe.

In [ ]:
calc = calc.assign(t=calc["date"].dt.year + calc["date"].dt.dayofyear / 365.25)

def doublement(donnees, nom):
    m = smf.ols("log10_calcul ~ t", data=donnees).fit()
    b, (bas, haut) = m.params["t"], m.conf_int().loc["t"]
    mois = lambda x: 12 * np.log10(2) / x
    return {"échantillon": nom, "n": int(m.nobs), "pente (log10/an)": round(b, 3),
            "doublement (mois)": round(mois(b), 1),
            "IC 95 % (mois)": f"{mois(haut):.1f} - {mois(bas):.1f}", "R²": round(m.rsquared, 2)}

pd.DataFrame([
    doublement(calc, "tous les modèles"),
    doublement(calc[calc["frontiere"]], "modèles de pointe"),
    doublement(calc[calc["pays"] == "États-Unis"], "États-Unis"),
    doublement(calc[calc["pays"] == "Chine"], "Chine"),
])

> **À interpréter.** En combien de mois la puissance de calcul double-t-elle ? Les intervalles de confiance des États-Unis et de la Chine se recouvrent-ils ? Pourquoi n'estime-t-on pas de pente pour la France seule (regarder `n`) ?

### A.5 Le retard exprimé en années

Pour chaque pays et chaque année, on prend le **record** de puissance de calcul atteint jusque-là, puis on cherche en quelle année les États-Unis avaient atteint ce même niveau. La différence est un retard en années ; une valeur négative signifie que le pays était en avance sur les États-Unis cette année-là. Le calcul repose sur des années entières : un écart d'un an peut venir d'un seul modèle.

In [ ]:
retard = annees_de_retard(par_pays)
tableau_retard = (retard.pivot(index="annee", columns="pays", values="retard_annees")
                  .reindex(columns=[p for p in ORDRE_PAYS if p not in ("États-Unis", "Autres")]))
tableau_retard.loc[tableau_retard.index >= 2018]

In [ ]:
fig, ax = plt.subplots()
for pays in tableau_retard.columns:
    s = tableau_retard[pays]
    s = s[s.index >= 2018]   # les années sans donnée restent vides : pas de ligne tracée par-dessus
    ax.plot(s.index, s.values, marker="o", markersize=5, color=COULEURS_PAYS[pays], label=pays,
            linewidth=3 if pays == "France" else 1.8)
ax.axhline(0, color="#8a8984", linewidth=1)
ax.set_title("Retard sur les États-Unis, en années de puissance de calcul")
ax.set_ylabel("Années de retard")
ax.legend(ncol=3)
fig.tight_layout(); fig.savefig(FIGURES / "04_retard_annees.png")

> **À interpréter.** La France rattrape-t-elle son retard ou décroche-t-elle ? Formuler le résultat en une phrase du type « en 2025, la France atteint le niveau que les États-Unis avaient en … ».

### A.6 Test de robustesse : la France sans Mistral

Hypothèse 2 : la présence française repose sur très peu d'acteurs. On regarde la concentration des modèles français par organisation.

In [ ]:
fr = par_pays.loc[par_pays["pays"] == "France"]
concentration = fr.groupby("Organization")["poids"].sum().sort_values(ascending=False)
print(f"{concentration.size} organisations françaises ; la première pèse {concentration.iloc[0] / concentration.sum():.0%} des modèles")
concentration.head(10)

In [ ]:
sans_mistral = par_pays.loc[~par_pays["Organization"].str.contains("Mistral", case=False, na=False)]
comparaison = pd.DataFrame({
    "avec Mistral": par_pays.loc[par_pays["pays"] == "France"].groupby("annee")["poids"].sum(),
    "sans Mistral": sans_mistral.loc[sans_mistral["pays"] == "France"].groupby("annee")["poids"].sum(),
}).fillna(0)
comparaison.loc[comparaison.index >= 2018]

> **À interpréter.** Que reste-t-il de la France sans Mistral ? Qu'est-ce que cela implique pour la robustesse de la position française ?

## Partie B — L'opacité des développeurs

### B.1 Qui publie quoi, et depuis quand ?

Pour chaque modèle, on sait si cinq informations ont été rendues publiques : puissance de calcul, nombre de paramètres, taille du jeu de données, coût, matériel. Le **score de transparence** compte ces informations (de 0 à 5).

*Deux limites à garder en tête :*

- **Ce qu'Epoch mesure, c'est l'information connue, pas l'information publiée par le développeur.** Pour un modèle aux poids ouverts, Epoch peut compter lui-même les paramètres ou estimer la puissance de calcul. Une valeur renseignée ne prouve donc pas que le développeur l'a divulguée. Le notebook 4 confronte cette mesure à celle de Stanford, qui note directement les divulgations des développeurs.
- **Délai d'estimation.** Epoch complète ses fiches au fil du temps : pour les modèles des derniers mois, une valeur manquante peut simplement signifier que l'estimation n'est pas encore faite. La baisse sur les toutes dernières années est donc en partie mécanique.

In [ ]:
indicateurs = list(INFOS_PUBLIABLES.values())
evolution = epoch.loc[epoch["annee"] >= 2015].groupby("annee")[indicateurs].mean()

fig, ax = plt.subplots()
noms = {"publie_calcul": "Puissance de calcul", "publie_parametres": "Paramètres",
        "publie_donnees": "Taille des données", "publie_materiel": "Matériel", "publie_cout": "Coût"}
couleurs = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
for (col, nom), c in zip(noms.items(), couleurs):
    ax.plot(evolution.index, evolution[col], color=c, label=nom)
ax.yaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.set_ylim(0, 1)
ax.set_title("Part des modèles notables dont l'information est publique")
ax.legend(ncol=3, loc="lower left")
fig.tight_layout(); fig.savefig(FIGURES / "05_opacite_evolution.png")

> **À interpréter.** Quelle information disparaît le plus vite ? À partir de quelle année la tendance s'inverse-t-elle ? Relier à la commercialisation des grands modèles.

### B.2 Ouvert n'est pas transparent ?

In [ ]:
tab = (epoch.loc[(epoch["annee"] >= 2020) & epoch["ouvert"].notna()]
       .assign(statut=lambda d: np.where(d["ouvert"], "poids ouverts", "fermé"))
       .groupby(["statut", "frontiere"])[indicateurs + ["score_transparence"]].mean()
       .rename(index={True: "pointe", False: "autre"}, level=1))
tab.style.format({c: "{:.0%}" for c in indicateurs} | {"score_transparence": "{:.2f}"})

> **À interpréter.** Les modèles ouverts publient-ils davantage leur puissance de calcul et leurs données d'entraînement ? L'écart est-il plus fort parmi les modèles de pointe ?

### B.3 Un modèle simple : qu'est-ce qui explique la publication de la puissance de calcul ?

Régression logistique de `publie_calcul` sur l'année, l'ouverture, le caractère de pointe et le pays. Les coefficients sont convertis en **rapports de chances** (odds ratios) : un rapport de 0,8 pour l'année signifie que les chances de publier sont multipliées par 0,8 chaque année, toutes choses égales par ailleurs.

In [ ]:
donnees_logit = (par_pays.loc[(par_pays["annee"] >= 2018) & (par_pays["poids"] == 1) & par_pays["ouvert"].notna()]
                 .assign(publie=lambda d: d["publie_calcul"].astype(int),
                         ouvert=lambda d: d["ouvert"].astype(int),
                         frontiere=lambda d: d["frontiere"].astype(int),
                         annees_depuis_2018=lambda d: d["annee"] - 2018))
# L'année est centrée sur 2018 : sans cela, la constante correspond à l'an 0
# et son exponentielle déborde.
logit = smf.logit("publie ~ annees_depuis_2018 + ouvert + frontiere + C(pays, Treatment('États-Unis'))",
                  data=donnees_logit).fit(disp=False)

resultats = pd.DataFrame({"rapport de chances": np.exp(logit.params),
                          "IC bas": np.exp(logit.conf_int()[0]),
                          "IC haut": np.exp(logit.conf_int()[1]),
                          "p-valeur": logit.pvalues}).round(3)
print(f"n = {int(logit.nobs)}, pseudo-R² = {logit.prsquared:.3f}")
resultats

> **À interpréter.** Le coefficient de l'année confirme-t-il l'hypothèse 3 ? L'effet de l'ouverture tient-il une fois contrôlé le pays ? Rappeler que cette régression décrit des corrélations : elle ne dit pas *pourquoi* un labo publie ou non.

### B.4 Avant / après l'AI Act (2 août 2025)

Comparaison **descriptive**. Attention au délai d'estimation d'Epoch (voir B.1) : les modèles postérieurs à août 2025 sont les plus récents, donc ceux dont les fiches sont les moins complètes. Une baisse après l'AI Act ne peut pas être lue comme un effet de la réglementation. La comparaison entre signataires et non-signataires, qui neutralise en partie ce biais, est dans le notebook 4.

In [ ]:
recent = epoch.loc[epoch["date"] >= "2024-01-01"]
recent.groupby("apres_ai_act")[indicateurs + ["score_transparence"]].agg(["mean", "count"]).T

## Synthèse — à rédiger

- Hypothèse 1 :
- Hypothèse 2 :
- Hypothèse 3 :
- Hypothèse 4 :

**Lien entre les deux parties.** Les modèles dont la puissance de calcul est inconnue sont absents des parties A.3 à A.5. Si ce sont surtout des modèles américains fermés de pointe, que devient le retard mesuré de la France ? Est-il sur- ou sous-estimé ?